In [1]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.2,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.180 pyspark-shell"
)

In [2]:
import os
import json
import time
import requests
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [3]:
spark = SparkSession.builder \
    .appName("SparkMinIOExample") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "Projeto_Final") \
    .config("spark.hadoop.fs.s3a.secret.key", "Projeto_Final") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

In [4]:
# ============================================================
# Função de coleta da API SPTrans
# ============================================================
def coletar_dados():
    API_TOKEN = "84dfde66637e13d4307f9408fc4d5c36474c7dc322310ef71f264dda3fe75704"
    LOGIN_URL = f"https://api.olhovivo.sptrans.com.br/v2.1/Login/Autenticar?token={API_TOKEN}"

    login_response = requests.post(LOGIN_URL)
    if login_response.status_code != 200:
        print(f"[ERRO] Falha no login: {login_response.status_code}")
        return []

    cookie_value = login_response.cookies.get("apiCredentials")
    if not cookie_value:
        print("[ERRO] Cookie não retornado pela API")
        return []

    regioes = {
        "zona_norte": [543, 614, 558, 2495, 709],
        "zona_sul": [1140, 59, 1977, 1318, 1318],
        "zona_leste": [2160, 1055, 2580, 1808],
        "zona_oeste": [689, 1376, 782, 472],
        "centro": [1523, 768, 2506, 1366]
    }

    registros = []
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for regiao, cls in regioes.items():
        for codigo_linha in cls:
            try:
                url = f"https://api.olhovivo.sptrans.com.br/v2.1/Previsao/Linha?codigoLinha={codigo_linha}"
                resp = requests.get(url, cookies={"apiCredentials": cookie_value})
                registros.append({
                    "regiao": regiao,
                    "codigo_linha": codigo_linha,
                    "timestamp": ts,
                    "raw_json": resp.text if resp.status_code == 200 else f"ERRO {resp.status_code}"
                })
            except Exception as e:
                registros.append({
                    "regiao": regiao,
                    "codigo_linha": codigo_linha,
                    "timestamp": ts,
                    "raw_json": f"ERRO: {e}"
                })

    return registros

In [5]:
# ============================================================
# Função para salvar no MinIO (camada RAW)
# ============================================================
def salvar_no_minio(dados):
    if not dados:
        print("[AVISO] Nenhum dado coletado nesta iteração.")
        return

    df = spark.createDataFrame(dados)
    (
        df.write
        .mode("append")
        .json("s3a://raw/sptrans/previsao/")
    )

In [6]:
# ============================================================
# Loop contínuo
# ============================================================
if __name__ == "__main__":
    INTERVALO_MINUTOS = 1  # tempo entre coletas

    while True:
        print(f"\n[INFO] Iniciando nova coleta às {datetime.now().strftime('%H:%M:%S')}...")
        dados = coletar_dados()
        salvar_no_minio(dados)
        print(f"[OK] Coleta concluída. Aguardando {INTERVALO_MINUTOS} minutos...\n")
        time.sleep(INTERVALO_MINUTOS * 60)


[INFO] Iniciando nova coleta às 02:48:47...
[OK] Coleta concluída. Aguardando 1 minutos...


[INFO] Iniciando nova coleta às 02:50:12...
[OK] Coleta concluída. Aguardando 1 minutos...



KeyboardInterrupt: 